In [1]:
# 공공데이터 약국 활용 예제 : 건강보험심사평가원_약국정보서비스

In [14]:
import requests
from bs4 import BeautifulSoup

service_key = '9f45f7a1696e595dddf89a923c94381ac985eb9256f89b343fb93bb0477d020b'

def getPharmacy(service_key : str, page_no : int = 1):
	"""공공 데이터에서 약국 정보를 가져오는 함수"""
	# End Point + 활용신청 상세기능정보 > 상세정보에 있는 주소
	url = 'https://apis.data.go.kr/B551182/pharmacyInfoService/getParmacyBasisList'
	params = {
		'ServiceKey' : service_key,
		'pageNo' : page_no #page라고 쓰면 pageNo가 없어도 기본값 1이 들어가서 결과가 나오긴 함
	}

	try:
		response = requests.get(url, params=params)

		response.raise_for_status()

		# 응답 결과가 xml 파일이어서 변수명이 res_xml로 함(강사 마음)
		res_xml = response.text

		soup = BeautifulSoup(res_xml, 'lxml-xml') # html이면 'lxml' 이고, xml이면 'lxml-xml로 쓴다.

		# 전체 약국수
		totalCount = soup.find('totalCount').text

		# 약국들 정보
		items = soup.find_all('item')

		yadmNms = []
		clCdNms = []
		sidoCdNms = []
		sgguCdNms = []
		addrs = []

		# 받아온 데이터들을 가져와서 리스트에 추가
		for item in items:
			yadmNms.append(item.find('yadmNm').text) # 병원명
			clCdNms.append(item.find('clCdNm').text) # 종별 코드명(약국)
			sidoCdNms.append(item.find('sidoCdNm').text) # 시도명
			sgguCdNms.append(item.find('sgguCdNm').text) # 시군구명
			addrs.append(item.find('addr').text) # 주소

		# 받아온 데이터를 딕셔너리로 변환
		res_dic = {
			'병원명' : yadmNms,
			'종별코드명' : clCdNms,
			'시도명' : sidoCdNms,
			'시군구명' : sgguCdNms,
			'주소' : addrs
		}

		# 최종 데이터가 있는 딕셔너리를 반환
		return res_dic

	except Exception as e:
		print(f"예외 발생 {e}")
		return {}

In [15]:
print(getPharmacy(service_key, 2))

{'병원명': ['명동 더서울약국', '행운약국', '행복한약국', '새서울약국', '북가좌수약국', '더조은약국', '큰동의약국', '즐거운약국', '땅콩약국', '새로운약국'], '종별코드명': ['약국', '약국', '약국', '약국', '약국', '약국', '약국', '약국', '약국', '약국'], '시도명': ['서울', '서울', '충남', '서울', '서울', '서울', '부산', '경기', '강원', '경기'], '시군구명': ['중구', '중랑구', '당진시', '구로구', '서대문구', '강동구', '부산진구', '안산상록구', '원주시', '성남중원구'], '주소': ['서울특별시 중구 명동길 27, 1~3층 (명동1가)', '서울특별시 중랑구 사가정로 386, 1층 101호 (면목동)', '충청남도 당진시 시청1로 75, (읍내동)', '서울특별시 구로구 가마산로 268, 대림역와이즈플레이스 108호 (구로동)', '서울특별시 서대문구 응암로 60, 경원빌딩 1층 (북가좌동)', '서울특별시 강동구 상암로 41, 1층 120, 120-1호 (암사동, 암사동양덱스빌)', '부산광역시 부산진구 진남로 554, 270동 A-109, A-110호 (양정동, 양정자이더샵SKVIEW 2단지)', '경기도 안산시 상록구 본이로 53, (본오동)', '강원특별자치도 원주시 북원로 2235-4, 116호 (단계동)', '경기도 성남시 중원구 성남대로 1141, 1층 (성남동, 둔전빌딩)']}


In [19]:
import pandas as pd
import time

df = pd.DataFrame({
	'병원명' : [],
	'종별코드명' : [],
	'시도명' : [],
	'시군구명' : [],
	'주소' : []
})

for i in range(1, 6):
	tmp_df = pd.DataFrame(getPharmacy(service_key, i))
	df = pd.concat([df, tmp_df])
	print(f"{i} 페이지가 로딩되었습니다.")
	time.sleep(1)

	print(df)

1 페이지가 로딩되었습니다.
        병원명 종별코드명 시도명    시군구명  \
0     서평택약국    약국  경기     평택시   
1      이수약국    약국  충북     영동군   
2     드림호약국    약국  인천  인천미추홀구   
3  365 열시약국    약국  광주    광주북구   
4      봄한약국    약국  강원     홍천군   
5      새싹약국    약국  서울     성북구   
6      세온약국    약국  대구    대구동구   
7       현약국    약국  서울     금천구   
8      동아약국    약국  부산    부산진구   
9    위드팜필약국    약국  서울    서대문구   

                                                  주소  
0                         경기도 평택시 포승읍 포승향남로 148, 148  
1                       충청북도 영동군 영동읍 난계로 1202, (영동읍)  
2        인천광역시 미추홀구 용정공원로83번길 59, 드림빌딩 1층 102호 (용현동)  
3                   광주광역시 북구 군왕로 297, 365 열시약국 (각화동)  
4                    강원특별자치도 홍천군 홍천읍 꽃뫼로 54-3, (홍천읍)  
5                    서울특별시 성북구 동소문로7길 4, 1층 (동소문동4가)  
6             대구광역시 동구 반야월로 205, 동호빌딩 108.109호 (동호동)  
7      서울특별시 금천구 가산디지털1로 149, 1층 103호 (가산동, 신한이노플렉스)  
8  부산광역시 부산진구 진남로 554, 270동 A-111, A-112호 (양정동, 양...  
9                           서울특별시 서대문구 신촌로 91, (창천동)  
2 페이지가 로딩되었습니다.

In [21]:
# 인덱스 번호 초기화
df = df.reset_index(drop=True)

In [23]:
# csv 파일로 저장
df.to_csv('약국 목록.csv', index=False, encoding='utf-8-sig')

In [25]:
# csv 파일 읽어오기
df2 = pd.read_csv('약국 목록.csv', encoding='utf-8')
print(df2)

         병원명 종별코드명 시도명    시군구명  \
0      서평택약국    약국  경기     평택시   
1       이수약국    약국  충북     영동군   
2      드림호약국    약국  인천  인천미추홀구   
3   365 열시약국    약국  광주    광주북구   
4       봄한약국    약국  강원     홍천군   
5       새싹약국    약국  서울     성북구   
6       세온약국    약국  대구    대구동구   
7        현약국    약국  서울     금천구   
8       동아약국    약국  부산    부산진구   
9     위드팜필약국    약국  서울    서대문구   
10  명동 더서울약국    약국  서울      중구   
11      행운약국    약국  서울     중랑구   
12     행복한약국    약국  충남     당진시   
13     새서울약국    약국  서울     구로구   
14    북가좌수약국    약국  서울    서대문구   
15     더조은약국    약국  서울     강동구   
16     큰동의약국    약국  부산    부산진구   
17     즐거운약국    약국  경기   안산상록구   
18      땅콩약국    약국  강원     원주시   
19     새로운약국    약국  경기   성남중원구   
20     둔산탑약국    약국  대전    대전서구   
21      초록약국    약국  경기     시흥시   
22    문산보룡약국    약국  경기     파주시   
23  청담장수 한약국    약국  서울     강남구   
24      대도약국    약국  서울     종로구   
25   메디팜정연약국    약국  서울     강남구   
26      진선약국    약국  서울     강남구   
27      국민약국    약국  서울     강서구   
28     더현대약국  

In [ ]:
# 경기에 있는 약국 수를 조회
print(f"경기에 있는 약국 수 : {len(df.loc[df['시도명'] == '경기'])}")


경기에 있는 약국 수 : 6


In [ ]:
# 결측치가 있는지 확인 : None, NaN, NaT
df.replace("", None)
print(df.isnull().sum())

병원명      0
종별코드명    0
시도명      0
시군구명     0
주소       0
dtype: int64
